[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB08_unet_montgomery.ipynb)

# NB08 · Lung segmentation with a U-Net (Montgomery CXR)
**Artificial Intelligence in Medical Imaging** · Day 3 · Lab 9 (42 min)

| | |
|---|---|
| Runtime | **GPU recommended** (T4). CPU works at 128 px with fewer epochs |
| Data | Montgomery County CXR set (138 images, 80 normal / 58 TB), downsampled to 256×256, split 96 / 21 / 21 |
| Checkpoints | merged mask is 0/1 with the image's shape · U-Net returns 1×256×256 · **test Dice computed once** |
| Tiers | **Core** = U-Net, loss, per-image Dice on val and test · **Plus** = bootstrap interval, best/worst overlays · **Challenge** = MONAI UNet or Kvasir-SEG |

**Learning goals.** (1) Prepare image/mask pairs correctly (merge masks, resize, same transform on both). (2) Build and train a small U-Net with a BCE + Dice loss. (3) Evaluate with **per-image Dice**, with an interval, and look at the worst cases. (4) Know the failure modes of segmentation code.

**Data licence note.** The Montgomery set comes from the U.S. National Library of Medicine. Its ReadMe asks users not to share the data outside their research group and to cite Candemir et al. (2014) and Jaeger et al. (2014). Use it for this course only; do not redistribute it.

In [ ]:
import os, random, time
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
SEED = 42; set_seed(SEED); Path("data").mkdir(exist_ok=True); Path("outputs").mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); GPU = device.type == "cuda"
EPOCHS = 30 if GPU else 8
RES = 256 if GPU else 128
print("torch", torch.__version__, "| device", device, "| training resolution", RES, "| epochs", EPOCHS)

## 1. Data: one small file from Moodle
The images and masks come from the NLM Montgomery set. The instructor downsampled them to 256×256 and packed them into a single small file, `montgomery_256.npz`, available **on Moodle for enrolled students only** (please do not share it: the dataset terms ask for it to stay within the research group).

**Download it from Moodle**, then run the next cell: in Colab it opens a file chooser (or upload it by hand to the folder `data/` in the file panel on the left).

The `.npz` holds: `images` (N×256×256, uint8), `masks` (merged left+right lung, 0/1), `tb` (label), `ids`, and fixed `train_idx`, `val_idx`, `test_idx` (96 / 21 / 21, stratified by TB label, seed 0).

In [ ]:
NPZ = Path("data/montgomery_256.npz")

def get_data():
    if not NPZ.exists():
        try:                                                        # Google Colab: opens a file chooser
            from google.colab import files
            print("Choose montgomery_256.npz (downloaded from Moodle) ...")
            up = files.upload()
            if "montgomery_256.npz" in up: Path("montgomery_256.npz").replace(NPZ)
        except ImportError: pass
    if not NPZ.exists():
        raise SystemExit("montgomery_256.npz not found. Download it from Moodle and place it in the folder data/ (in Colab: file panel on the left, folder data, upload), then run this cell again.")
    return dict(np.load(NPZ, allow_pickle=False)), "local file"

D, source = get_data()
print("data source:", source, "| images", D["images"].shape, "| masks", D["masks"].shape)

### Checkpoint 1 · the masks are merged, binary and aligned with the images

In [ ]:
imgs, masks = D["images"], D["masks"]
assert imgs.shape == masks.shape == (len(imgs), 256, 256), "image and mask must have the same shape"
assert set(np.unique(masks)) <= {0, 1}, "masks must contain only 0 and 1"
frac = masks.mean(axis=(1, 2)); print(f"lung area as a share of the image: min {frac.min():.2f}, median {np.median(frac):.2f}, max {frac.max():.2f}")
assert np.median(frac) > 0.18 and frac.min() > 0.05, "masks look too small: ONE lung only? merge left and right masks (two lungs: median about 0.25; one lung: about 0.13)"
tr, va, te = D["train_idx"], D["val_idx"], D["test_idx"]
assert len(set(tr) & set(va)) == len(set(tr) & set(te)) == len(set(va) & set(te)) == 0, "splits overlap"
print("split sizes:", len(tr), len(va), len(te)); print("Checkpoint 1 passed")
fig, ax = plt.subplots(1, 4, figsize=(12, 3.2))
for k, i in enumerate(tr[:2]):
    ax[2*k].imshow(imgs[i], cmap="gray"); ax[2*k].set_title(str(D["ids"][i])); ax[2*k+1].imshow(imgs[i], cmap="gray"); ax[2*k+1].imshow(np.ma.masked_equal(masks[i], 0), cmap="autumn", alpha=0.5); ax[2*k+1].set_title("lung mask")
for a_ in ax: a_.axis("off")
plt.show()

## 2. Tensors and a **joint** augmentation
Whatever geometric transform you apply to an image you must apply **identically** to its mask (flip the image but not the mask = the classic silent bug). Intensity changes go on the image only. We use a random horizontal shift and a brightness/contrast jitter, applied to a *batch* to keep it fast.

In [ ]:
def to_tensors(idx, res=RES):
    x = torch.from_numpy(imgs[idx]).float().div(255).unsqueeze(1); y = torch.from_numpy(masks[idx]).float().unsqueeze(1)
    if res != 256: x = F.interpolate(x, size=(res, res), mode="bilinear", align_corners=False); y = (F.interpolate(y, size=(res, res), mode="nearest") > 0.5).float()
    return x, y
Xtr, Ytr = to_tensors(tr); Xva, Yva = to_tensors(va); Xte, Yte = to_tensors(te)

def augment(x, y, gen):
    """Same geometric transform for image and mask; intensity jitter for the image only."""
    B = x.shape[0]; shift = int(torch.randint(-int(0.06 * x.shape[-1]), int(0.06 * x.shape[-1]) + 1, (1,), generator=gen))
    x, y = torch.roll(x, shift, dims=-1), torch.roll(y, shift, dims=-1)                 # joint geometry: same shift for image and mask
    a = 1 + 0.2 * (torch.rand(B, 1, 1, 1, generator=gen) - 0.5); b = 0.1 * (torch.rand(B, 1, 1, 1, generator=gen) - 0.5)
    return (x * a.to(x.device) + b.to(x.device)).clamp(0, 1), y

g = torch.Generator().manual_seed(0); xa, ya = augment(Xtr[:4].clone(), Ytr[:4].clone(), g)
# image and mask must still be aligned after augmentation: compare the lung-mask centroid with the intensity profile (the mask moved with the image)
assert xa.shape == Xtr[:4].shape and ya.shape == Ytr[:4].shape and set(ya.unique().tolist()) <= {0.0, 1.0}
print("augmentation keeps the mask binary and the shapes unchanged")

## 3. The U-Net
Skeleton from the slides: two levels of down-sampling, a bottleneck, and **skip connections** that concatenate encoder features into the decoder. Output = **logits** (no sigmoid inside the model).

```
input 1×H×W → e1 (16) → pool → e2 (32) → pool → mid (64) → up → cat(e2) → d2 (32) → up → cat(e1) → d1 (16) → 1×1 conv → 1×H×W
```
**TODO 1.** Complete the decoder part of `forward`: upsample, concatenate with the skip, apply the block. Hint: `torch.cat([up, skip], dim=1)`; the channel counts in `__init__` already match.

In [ ]:
def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1, self.e2, self.mid = block(1, 16), block(16, 32), block(32, 64)
        self.pool = nn.MaxPool2d(2)
        self.up2, self.d2 = nn.ConvTranspose2d(64, 32, 2, stride=2), block(64, 32)
        self.up1, self.d1 = nn.ConvTranspose2d(32, 16, 2, stride=2), block(32, 16)
        self.out = nn.Conv2d(16, 1, 1)
    def forward(self, x):
        s1 = self.e1(x); s2 = self.e2(self.pool(s1)); m = self.mid(self.pool(s2))
        # TODO 1a: d2 = self.d2( concatenate [self.up2(m), s2] along the channel dimension )
        d2 = None
        # TODO 1b: d1 = self.d1( concatenate [self.up1(d2), s1] )
        d1 = None
        return self.out(d1)                         # logits

In [ ]:
net = UNet().to(device)
with torch.no_grad(): o = net.eval()(torch.zeros(1, 1, 256, 256, device=device))
print("output shape:", tuple(o.shape), "| parameters:", f"{sum(p.numel() for p in net.parameters()):,}")
assert o.shape == (1, 1, 256, 256), "the U-Net must return one logit per pixel: 1x1x256x256"
with torch.no_grad(): assert net(torch.zeros(2, 1, RES, RES, device=device)).shape == (2, 1, RES, RES)
print("Checkpoint 2 passed")

## 4. Loss and metric
**Dice** for one image: `2|P∩G| / (|P|+|G|)`, with P the *thresholded* prediction (probability > 0.5) and G the ground truth. **Convention:** if both are empty the Dice is 0/0; we define it as 1 (perfect agreement). Here every image has lungs, so it does not occur.

**Loss** = BCE on logits + soft Dice, `1 − (2·Σ(p·g)+ε) / (Σp+Σg+ε)` with `p = sigmoid(logits)` (probabilities, not hard 0/1: the loss must be differentiable).

> Two classic bugs: applying `sigmoid` twice, and computing the *metric* on probabilities or logits instead of the thresholded mask.

**TODO 2.** Complete `soft_dice_loss` and `dice_score`.

In [ ]:
def soft_dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    # TODO 2a: per-image intersection  sum(p * target)  and  den = sum(p) + sum(target)   (sum over dims 1,2,3)
    inter, den = 0, 1
    return (1 - (2 * inter + eps) / (den + eps)).mean()

def seg_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + soft_dice_loss(logits, target)

def dice_score(pred_mask, target, eps=1e-8):
    """Per-image Dice of binary masks (B,1,H,W) -> (B,). Both empty -> 1."""
    # TODO 2b: inter and den as above (pred_mask is already 0/1); return 2*inter/den, but 1 where den == 0
    return torch.ones(pred_mask.shape[0])

In [ ]:
# Dice by hand on the slide's cases (4x4 toy images) -> expected hard Dice values
G = torch.zeros(1, 1, 4, 4); G[..., 1:3, 1:3] = 1                       # 4 pixels of "lung"
P_shift = torch.zeros(1, 1, 4, 4); P_shift[..., 1:3, 2:4] = 1          # shifted by one column: overlap 2 of 4+4
P_empty = torch.zeros(1, 1, 4, 4); P_all = torch.ones(1, 1, 4, 4)
print("shifted:", dice_score(P_shift, G).item(), "| empty prediction:", dice_score(P_empty, G).item(), "| everything lung:", dice_score(P_all, G).item(), "| both empty:", dice_score(P_empty, P_empty).item())
assert abs(dice_score(P_shift, G).item() - 0.5) < 1e-6 and dice_score(P_empty, G).item() < 1e-6
assert abs(dice_score(P_all, G).item() - 2*4/(16+4)) < 1e-6 and dice_score(P_empty, P_empty).item() == 1.0
assert dice_score(G, G).item() > 0.999
# the soft loss is small for a confident correct logit map and large for a wrong one
L_good = soft_dice_loss((G * 20 - 10), G).item(); L_bad = soft_dice_loss(((1 - G) * 20 - 10), G).item()
assert L_good < 0.1 and L_bad > 0.8, (L_good, L_bad)
print("Checkpoint 3 passed: Dice and loss behave as expected")

## 5. Train
Adam, `lr=1e-3`, batch size 8, augmentation on the training set only. After each epoch we print the **validation** Dice (mean of the per-image Dice). We keep the weights with the best validation Dice (choice made on validation only).

In [ ]:
def predict_mask(model, x, bs=16):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(x), bs): out.append((torch.sigmoid(model(x[i:i+bs].to(device))) > 0.5).float().cpu())
    return torch.cat(out)

def train_unet(epochs=EPOCHS, lr=1e-3, bs=8, seed=SEED):
    set_seed(seed); model = UNet().to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    gen = torch.Generator().manual_seed(seed); best, best_state, hist = -1, None, []
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr), generator=gen)
        for i in range(0, len(perm), bs):
            b = perm[i:i+bs]
            if len(b) < 2: continue                                  # BatchNorm needs more than one image
            xb, yb = augment(Xtr[b].clone(), Ytr[b].clone(), gen)
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); seg_loss(model(xb), yb).backward(); opt.step()
        d = dice_score(predict_mask(model, Xva), Yva).mean().item(); hist.append(d)
        if d > best: best, best_state = d, {k: v.detach().clone() for k, v in model.state_dict().items()}
        if ep % 3 == 0 or ep == epochs - 1: print(f"epoch {ep+1:2d}/{epochs} | validation Dice {d:.3f}")
    model.load_state_dict(best_state); return model, hist

t0 = time.time(); model, hist = train_unet(); print(f"best validation Dice {max(hist):.3f} | {(time.time()-t0)/60:.1f} min")
plt.plot(range(1, len(hist) + 1), hist, marker="o"); plt.xlabel("epoch"); plt.ylabel("validation Dice"); plt.show()

### Per-image validation Dice and the worst cases
The mean hides failures. Look at the distribution and at the **worst** images.

In [ ]:
pm_val = predict_mask(model, Xva); dv = dice_score(pm_val, Yva).numpy()
print(f"validation Dice: mean {dv.mean():.3f}, median {np.median(dv):.3f}, min {dv.min():.3f}, max {dv.max():.3f}")
plt.hist(dv, bins=15); plt.xlabel("per-image Dice (validation)"); plt.ylabel("images"); plt.show()
assert dv.mean() > 0.5, "the model should segment clearly better than chance; check the loss, the sigmoid and the mask/image alignment"

## 6. Test set, once
Report the per-image Dice on the 21 test images. Same one-shot rule as NB06: no changes after you look.

In [ ]:
_test_runs = []
def test_report(model):
    """THE ONLY CELL THAT TOUCHES THE TEST SET. Run it once."""
    _test_runs.append(1)
    if len(_test_runs) > 1: print("!! the test set has been used", len(_test_runs), "times")
    pm = predict_mask(model, Xte); return dice_score(pm, Yte).numpy(), pm
dt, pm_test = test_report(model)
print(f"TEST Dice: mean {dt.mean():.3f} | median {np.median(dt):.3f} | min {dt.min():.3f} | max {dt.max():.3f} | n = {len(dt)}")

In [ ]:
assert len(_test_runs) == 1, "the test Dice must be computed exactly once (restart and run all)"
assert len(dt) == len(te)
print("Checkpoint 4 passed: test Dice computed once")

**Reflection questions.**
1. Validation vs test Dice with only 21 images each: how much do you trust the difference? (See the interval below.)
2. Is a mean Dice of 0.95 "good"? What does a Dice of 0.95 still allow in terms of clinical error (boundary, small structures)?
3. Which images are hardest? Look at the worst case: does it share a property (small lungs, device, severe disease)? Remember the dataset comes from a TB screening programme.

## Plus · bootstrap interval and best/worst overlays
**TODO (Plus).** Bootstrap the *mean* per-image Dice on the test set (resample images with replacement, 1,000 times, 95% percentile interval). Then plot the 3 best and 3 worst test images: image, ground truth contour and prediction contour.

In [ ]:
# [PLUS]
def bootstrap_mean_ci(values, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); n = len(values)
    # TODO (Plus): means = [mean of a bootstrap resample of `values`] repeated n_boot times
    means = [values.mean()]
    return np.percentile(means, [2.5, 97.5])
lo, hi = bootstrap_mean_ci(dt); print(f"test Dice {dt.mean():.3f}, 95% interval [{lo:.3f}, {hi:.3f}]")

# TODO (Plus): order = np.argsort(dt); plot the 3 worst and 3 best test images with the ground-truth contour (green)
# and the prediction contour (red): ax.contour(mask, levels=[0.5], colors="lime")
...

## Challenge
Choose **one**:
- **MONAI UNet.** `pip install monai` and use `monai.networks.nets.UNet(spatial_dims=2, in_channels=1, out_channels=1, channels=(16, 32, 64, 128), strides=(2, 2, 2), num_res_units=2)` with the same loss and data. Compare Dice, parameters and training time with the hand-made U-Net.
- **Kvasir-SEG polyps.** Colour endoscopy images with polyp masks (about 1,000 images, 46 MB). Check the dataset terms before downloading, resize to 256, adapt `in_channels` to 3 and report Dice with an interval. Why is Dice lower than for lungs?

Write 5 lines: what you changed, the numbers (with intervals), and one thing that surprised you.

In [ ]:
# [CHALLENGE]
# Challenge workspace: free-form. Reuse train_unet, seg_loss, dice_score and the one-shot rule.
...

## Summary
- Segmentation data prep is where silent bugs live: merge masks, same transform for image and mask, binary 0/1.
- A U-Net = encoder, bottleneck, decoder, **skip connections**; output logits, train with BCE + soft Dice, **evaluate with thresholded Dice per image**.
- Report distribution and interval, not just the mean; look at the worst cases.
- Test set once.

*Next:* Day 4 (explainability and project work).